In [2]:
# Импортируем алхимию и создаем соединение
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np

load_dotenv()

POSTGRES_USER = os.getenv('POSTGRES_USER')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')
POSTGRES_DB = os.getenv('POSTGRES_DB')
POSTGRES_HOST= os.getenv('POSTGRES_HOST', 'localhost')

conn = create_engine(f'postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:5432/{POSTGRES_DB}')


In [3]:
sql = '''
    SELECT 
        *
    FROM raw.extract_api_jobs
'''

df = pd.read_sql(sql, conn)
df

,id,endpoint,request_payload,response_data,status_code,loaded_at
0,1,MarketDataService/GetCandles,"{'to': '2026-06-30T06:08:41.455247+00:00', 'fi...","{'candles': [{'low': {'nano': 230000000, 'unit...",200,2026-06-30 06:08:41.585007+00:00
1,2,MarketDataService/GetCandles,"{'to': '2026-06-30T06:39:42.378524+00:00', 'fi...","{'candles': [{'low': {'nano': 0, 'units': '297...",200,2026-06-30 06:39:42.553008+00:00
2,3,MarketDataService/GetCandles,"{'to': '2026-07-01T05:15:45.774673+00:00', 'fi...","{'candles': [{'low': {'nano': 890000000, 'unit...",200,2026-07-01 05:15:45.932152+00:00
3,4,MarketDataService/GetCandles,"{'to': '2026-07-02T05:15:47.038426+00:00', 'fi...","{'candles': [{'low': {'nano': 850000000, 'unit...",200,2026-07-02 05:15:51.891103+00:00
4,5,MarketDataService/GetCandles,"{'to': '2026-07-02T05:15:49.436653+00:00', 'fi...","{'candles': [{'low': {'nano': 850000000, 'unit...",200,2026-07-02 05:15:54.170052+00:00
5,6,MarketDataService/GetCandles,"{'to': '2026-07-03T05:46:40.445665+00:00', 'fi...","{'candles': [{'low': {'nano': 670000000, 'unit...",200,2026-07-03 05:46:44.184541+00:00
6,7,MarketDataService/GetCandles,"{'to': '2026-07-03T06:38:12.709434+00:00', 'fi...","{'candles': [{'low': {'nano': 610000000, 'unit...",200,2026-07-03 06:38:15.254333+00:00
7,8,MarketDataService/GetCandles,"{'to': '2026-07-03T06:38:30.390869+00:00', 'fi...","{'candles': [{'low': {'nano': 610000000, 'unit...",200,2026-07-03 06:38:32.991671+00:00
8,9,InstrumentsService/GetAssets,"{'instrumentType': 'INSTRUMENT_TYPE_SHARE', 'i...",{'assets': [{'uid': 'b6a73950-20a8-46c7-8b49-9...,200,2026-07-03 06:38:53.395885+00:00
9,10,InstrumentsService/GetAssets,"{'instrumentType': 'INSTRUMENT_TYPE_SHARE', 'i...",{'assets': [{'uid': 'b6a73950-20a8-46c7-8b49-9...,200,2026-07-03 06:38:55.969439+00:00


In [11]:
sql = '''
with success_jobs as (
	select
		id,
		jsonb_array_length(response_data -> 'candles') AS resp_candles_cnt,
		(request_payload #>> '{from}')::date as req_from_dt,
		(request_payload #>> '{to}')::date as req_to_dt,
		request_payload #>> '{figi}' as req_figi,
		request_payload #>> '{interval}' as req_interval,
		status_code,
		loaded_at as db_loaded_dttm
	from raw.extract_api_jobs
	where endpoint = 'MarketDataService/GetCandles'
	and status_code = 200
)
select distinct
	c.date_id as from_dt,
	sj.req_to_dt as to_dt,
    sj.req_figi as figi,
    sj.req_interval as interval
from dim_calendar c
left join success_jobs sj on c.date_id = sj.req_from_dt
where 1=1
and c.date_id <= current_date
and c.date_id > current_date - interval '30 days'
order by 1

'''

df = pd.read_sql(sql, conn)
df

,from_dt,to_dt,figi,interval
0,2026-06-06,None,NaN,NaN
1,2026-06-07,None,NaN,NaN
2,2026-06-08,None,NaN,NaN
3,2026-06-09,None,NaN,NaN
4,2026-06-10,None,NaN,NaN
5,2026-06-11,None,NaN,NaN
6,2026-06-12,None,NaN,NaN
7,2026-06-13,None,NaN,NaN
8,2026-06-14,None,NaN,NaN
9,2026-06-15,None,NaN,NaN
